# Phase 1 Ablation Training on Kaggle
Mục tiêu: Huấn luyện các biến thể kiến trúc ít rủi ro `B1a`, `B1b`, `C1` theo kịch bản Phase 1.

> **LƯU Ý QUAN TRỌNG:**
> Bạn cần **PUSH các thay đổi mới nhất** (Hybrid Loss, AMP, Data Augmentation) trong thư mục `ablation` lên GitHub trước khi chạy Kaggle Notebook này, vì notebook sẽ clone repo trực tiếp từ GitHub.

In [ ]:
!pip install torchaudio tensorboard soundfile pandas tqdm einops pesq pystoi

import os
import shutil

# Clone repo (Thay đổi URL nếu bạn dùng repo cá nhân/private)
GIT_REPO = "https://github.com/hoxuanphu/Pdeepvqe.git"
REPO_DIR = "/kaggle/working/Pdeepvqe"

if not os.path.exists(REPO_DIR):
    !git clone {GIT_REPO} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Thư mục làm việc: {os.getcwd()}")

## 1. Tải và xử lý VCTK-DEMAND Dataset
Kaggle có tốc độ mạng rất nhanh, việc tải 4 file zip từ ĐH Edinburgh chỉ mất 1-2 phút.

In [ ]:
import zipfile

data_dir = "/kaggle/working/data/voicebank-demand"
os.makedirs(data_dir, exist_ok=True)

datasets = [
    ("clean_trainset_28spk_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip"),
    ("noisy_trainset_28spk_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip"),
    ("clean_testset_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip"),
    ("noisy_testset_wav.zip", "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip")
]

for filename, url in datasets:
    local_zip = os.path.join(data_dir, filename)
    if not os.path.exists(local_zip):
        print(f"Đang tải {filename}...")
        os.system(f"wget -q --show-progress {url} -O {local_zip}")
    
    extract_folder = local_zip.replace(".zip", "")
    if not os.path.exists(extract_folder):
        print(f"Đang giải nén {filename}...")
        with zipfile.ZipFile(local_zip, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
        print(f"Hoàn tất giải nén {filename}.")

print("Dữ liệu đã sẵn sàng!")

## 2. Tạo Manifest (CSV)
Tạo các file `train.csv`, `valid.csv`, `test.csv` để `train_ablation.py` đọc cấu trúc metadata của dữ liệu.

In [ ]:
import glob
import pandas as pd

def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(os.path.join(clean_dir, "*.wav")))
    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    assert len(clean_files) == len(noisy_files), "Số lượng file không khớp!"
    
    data = []
    for c, n in zip(clean_files, noisy_files):
        filename = os.path.basename(c)
        data.append({
            "ID": filename.replace(".wav", ""),
            "clean_wav": os.path.abspath(c),
            "noisy_wav": os.path.abspath(n)
        })
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Đã tạo {output_csv} với {len(df)} mẫu.")

base_dir = data_dir
create_csv(f"{base_dir}/clean_trainset_28spk_wav", f"{base_dir}/noisy_trainset_28spk_wav", f"{base_dir}/train_full.csv")
create_csv(f"{base_dir}/clean_testset_wav", f"{base_dir}/noisy_testset_wav", f"{base_dir}/test.csv")

# Split 90/10 cho train/valid
df_train_full = pd.read_csv(f"{base_dir}/train_full.csv")
df_train_full = df_train_full.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df_train_full) * 0.90)
df_train_full.iloc[:split_idx].to_csv(f"{base_dir}/train.csv", index=False)
df_train_full.iloc[split_idx:].to_csv(f"{base_dir}/valid.csv", index=False)
print("Hoàn tất tạo metadata!")

## 3. Huấn luyện Phase 1 (B1a, B1b, C1)
Train lần lượt các biến thể ablation với script `ablation/train_ablation.py` theo đúng Phase 1 Plan.
Kết quả checkpoints sẽ lưu tại `/kaggle/working/runs`.

In [ ]:
configs_to_train = ['B1a', 'B1b', 'C1']
epochs = 100
batch_size = 4  # Dùng batch 4 để công bằng với baseline

train_csv = f"{data_dir}/train.csv"
valid_csv = f"{data_dir}/valid.csv"
output_base = "/kaggle/working/runs/ablation"

import subprocess
import sys

for cfg in configs_to_train:
    print(f"\n{'='*60}")
    print(f" Bắt đầu train config: {cfg} ")
    print(f"{'='*60}\n")
    
    out_dir = f"{output_base}/{cfg}"
    
    cmd = [
        sys.executable, "ablation/train_ablation.py",
        "--config-id", cfg,
        "--train-manifest", train_csv,
        "--valid-manifest", valid_csv,
        "--output-dir", out_dir,
        "--epochs", str(epochs),
        "--batch-size", str(batch_size),
        "--num-workers", "2",
        "--early-stop-patience", "15",
        "--data-parallel"  # Tận dụng DataParallel nếu có 2 GPUs (Kaggle T4 x2)
    ]
    
    # Tự động Resume nếu có checkpoint cũ
    last_pt = f"{out_dir}/last.pt"
    if os.path.exists(last_pt):
        print(f"\n>>> Tìm thấy {last_pt}, tự động RESUME... <<<\n")
        cmd.extend(["--resume", last_pt])
    
    # Chạy process và in output realtime
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
    process.stdout.close()
    process.wait()
    
    if process.returncode != 0:
        print(f"\n[LỖI] Train {cfg} thất bại với mã lỗi {process.returncode}!")
        break
    else:
        print(f"\n=> Hoàn tất train {cfg}!")
        print("Checkpoint được lưu tại:", out_dir)

## 4. Zip kết quả lại để tải về dễ dàng
Sau khi train xong, zip thư mục `runs` lại thành 1 file để tải về nhanh chóng.

In [ ]:
import shutil

zip_path = "/kaggle/working/phase1_ablation_results"
shutil.make_archive(zip_path, 'zip', output_base)
print(f"Đã nén toàn bộ kết quả vào {zip_path}.zip")
print("Bạn có thể vào tab Data > Output bên phải giao diện Kaggle để tải file này về!")